# Dynamic Light Scattering, Zeta Potential, and Growth Inhibition Measurements for Self-Associating Amphiphiles Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This dataset package describes experimental data for a 32-member library of self-associating amphiphiles, including dynamic light scattering (DLS), zeta potential, and growth inhibition assay summaries. In this notebook, we focus on loading the Croissant metadata, inspecting the available dataset structure, and preparing reusable code that works with record sets and fields by their `@id` values.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

A key rule in this notebook is that all dataset entities should be handled using their `@id` values whenever possible. This is especially important for record sets and fields.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.tnk3-xhn0/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata

# Convert metadata to a JSON-serializable object for inspection
metadata_json = metadata_obj.to_json()

dataset_name = getattr(metadata_obj, 'name', metadata_json.get('name', 'Unknown dataset'))
dataset_description = getattr(metadata_obj, 'description', metadata_json.get('description', ''))

print(f"Dataset name: {dataset_name}")
print(f"Description: {dataset_description}")
print(f"Croissant URL: {url}")

In [ ]:
# High-level metadata summary
summary_fields = {
    '@id': metadata_json.get('@id'),
    'identifier': metadata_json.get('identifier'),
    'version': metadata_json.get('version'),
    'license': metadata_json.get('license'),
    'datePublished': metadata_json.get('datePublished'),
    'keywords_count': len(metadata_json.get('keywords', [])),
    'distribution_count': len(metadata_json.get('distribution', [])),
    'recordSet_count': len(metadata_json.get('recordSet', [])),
}

pd.DataFrame([summary_fields])

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant package provided for this dataset currently reports an empty `recordSet` list in the metadata snapshot. Because of that, this section is written defensively: it first inspects the metadata, then attempts record access only if record sets are available.

This approach makes the notebook reusable for future versions of the dataset where record sets may be added or exposed.

In [ ]:
# Inspect top-level metadata keys
top_level_keys = sorted(metadata_json.keys())
print('Top-level metadata keys:')
print(top_level_keys)

In [ ]:
# Extract record set metadata safely from the JSON representation
record_sets_meta = metadata_json.get('recordSet', []) or []

print(f"Number of record sets declared in metadata: {len(record_sets_meta)}")

record_set_rows = []
for rs in record_sets_meta:
    record_set_rows.append({
        '@id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description'),
        'field_count': len(rs.get('field', []) or [])
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

In [ ]:
# Display fields for each record set using their @id values
for rs in record_sets_meta:
    record_set_id = rs.get('@id')
    print(f"\nRecord set @id: {record_set_id}")
    print(f"Record set name: {rs.get('name')}")
    fields = rs.get('field', []) or []
    print(f"Field count: {len(fields)}")
    
    field_rows = []
    for field in fields:
        field_rows.append({
            '@id': field.get('@id'),
            'name': field.get('name'),
            'description': field.get('description'),
            'dataType': field.get('dataType'),
            'source': field.get('source')
        })
    
    if field_rows:
        display(pd.DataFrame(field_rows))
    else:
        print('No fields declared for this record set.')

The template below follows the example style directly and demonstrates how to iterate through records for a specific record set `@id`. It runs only if at least one record set is available.

In [ ]:
# Example record iteration using a record set @id
if record_sets_meta:
    example_record_set_id = record_sets_meta[0].get('@id')
    print(f"Iterating over records from record set @id: {example_record_set_id}")
    
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i >= 2:
            break
else:
    print('No record sets are available in the metadata, so no records can be iterated at this time.')

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

To stay consistent with Croissant best practices, this section collects record set `@id` values dynamically and stores each loaded table under its record set `@id`.

In [ ]:
# Extract data from each record set using their @id values
record_set_ids = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} rows from record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

print(f"Available DataFrames: {list(dataframes.keys())}")

In [ ]:
# Inspect the first available DataFrame, if any
if dataframes:
    example_record_set_id = next(iter(dataframes.keys()))
    print(f"Example record set @id: {example_record_set_id}")
    print('Columns:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No tabular record sets were loaded. This is expected if the current Croissant metadata does not expose recordSet entries.')

Because this dataset snapshot contains rich top-level metadata even when record sets are unavailable, we can still build useful analysis tables from the package-level fields.

In [ ]:
# Build a package-level DataFrame from selected metadata fields
package_df = pd.DataFrame({
    'keyword': metadata_json.get('keywords', [])
})

package_df.head()

In [ ]:
# Build a DataFrame for distributions, referenced by @id
distribution_rows = []
for item in metadata_json.get('distribution', []) or []:
    distribution_rows.append({
        '@id': item.get('@id') if isinstance(item, dict) else item,
    })

distributions_df = pd.DataFrame(distribution_rows)
distributions_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Since the current metadata snapshot does not include accessible record sets, this section demonstrates two complementary EDA paths:

1. **A generic Croissant record-set workflow** that activates automatically when record sets are available.
2. **A metadata-level analysis** using package information such as keywords, distributions, biases, limitations, and use cases.

In [ ]:
# Generic numeric EDA for the first loaded record set with numeric columns
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for record_set_id, df in dataframes.items():
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    non_numeric_candidates = [c for c in df.columns.tolist() if c not in numeric_candidates]
    if numeric_candidates:
        selected_record_set_id = record_set_id
        numeric_field_id = numeric_candidates[0]
        group_field_id = non_numeric_candidates[0] if non_numeric_candidates else None
        break

if selected_record_set_id is not None:
    working_df = dataframes[selected_record_set_id].copy()
    threshold = working_df[numeric_field_id].median()
    filtered_df = working_df[working_df[numeric_field_id] > threshold].copy()
    print(f"Selected record set @id: {selected_record_set_id}")
    print(f"Numeric field selected: {numeric_field_id}")
    print(f"Threshold used: {threshold}")
    print(f"Filtered rows: {len(filtered_df)}")
    
    if filtered_df[numeric_field_id].std() not in [0, None] and pd.notna(filtered_df[numeric_field_id].std()):
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print('Selected numeric field does not have enough variance for normalization.')
    
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=True).reset_index()
        print(f"Grouped summary by field: {group_field_id}")
        display(grouped_df.head())
else:
    print('No loaded record set with numeric columns is available for tabular EDA.')

In [ ]:
# Metadata-level EDA: analyze text lists exposed by the dataset package
metadata_list_lengths = pd.DataFrame([
    {'field': 'keywords', 'count': len(metadata_json.get('keywords', []) or [])},
    {'field': 'distribution', 'count': len(metadata_json.get('distribution', []) or [])},
    {'field': 'author', 'count': len(metadata_json.get('author', []) or [])},
    {'field': 'funding', 'count': len(metadata_json.get('funding', []) or [])},
    {'field': 'dataBiases', 'count': len(metadata_json.get('dataBiases', []) or [])},
    {'field': 'dataLimitations', 'count': len(metadata_json.get('dataLimitations', []) or [])},
    {'field': 'dataUseCases', 'count': len(metadata_json.get('dataUseCases', []) or [])}
])

metadata_list_lengths

In [ ]:
# Example filtering and normalization on metadata-derived counts
numeric_field_id = 'count'
threshold = 3
filtered_metadata_df = metadata_list_lengths[metadata_list_lengths[numeric_field_id] > threshold].copy()

print(f"Filtered rows with {numeric_field_id} > {threshold}:")
display(filtered_metadata_df)

if filtered_metadata_df[numeric_field_id].std() not in [0, None] and pd.notna(filtered_metadata_df[numeric_field_id].std()):
    filtered_metadata_df[f'{numeric_field_id}_normalized'] = (
        (filtered_metadata_df[numeric_field_id] - filtered_metadata_df[numeric_field_id].mean()) /
        filtered_metadata_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_metadata_df[[numeric_field_id, f'{numeric_field_id}_normalized']])
else:
    print('Not enough variance to normalize metadata counts.')

In [ ]:
# Group metadata-derived information by simple categories
categorized_rows = []
for field_name in ['keywords', 'distribution', 'author', 'funding', 'dataBiases', 'dataLimitations', 'dataUseCases']:
    category = 'provenance' if field_name in ['author', 'funding', 'distribution'] else 'content'
    categorized_rows.append({
        'group_field': category,
        'field_id': field_name,
        'count': len(metadata_json.get(field_name, []) or [])
    })

categorized_df = pd.DataFrame(categorized_rows)
grouped_df = categorized_df.groupby('group_field', as_index=False)['count'].mean()
grouped_df

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We include visualizations for both metadata-derived summaries and, when available, record-set tables loaded via `mlcroissant`.

In [ ]:
# Visualization 1: metadata list sizes
plt.figure(figsize=(10, 5))
sns.barplot(data=metadata_list_lengths.sort_values('count', ascending=False), x='field', y='count', palette='viridis')
plt.title('Counts of key metadata lists in the Croissant package')
plt.xlabel('Metadata field')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: keyword lengths
if not package_df.empty:
    keyword_lengths_df = package_df.copy()
    keyword_lengths_df['length'] = keyword_lengths_df['keyword'].astype(str).str.len()
    
    plt.figure(figsize=(8, 5))
    sns.histplot(keyword_lengths_df['length'], bins=10, kde=True, color='steelblue')
    plt.title('Distribution of keyword text lengths')
    plt.xlabel('Keyword length')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print('No keywords available for visualization.')

In [ ]:
# Visualization 3: if a tabular record set with numeric columns exists, show a histogram
if selected_record_set_id is not None and numeric_field_id is not None:
    plot_df = dataframes[selected_record_set_id].copy()
    
    plt.figure(figsize=(8, 5))
    sns.histplot(plot_df[numeric_field_id].dropna(), bins=20, kde=True, color='darkorange')
    plt.title(f'Distribution of numeric field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print('No tabular numeric field is available for record-level visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used `mlcroissant` to load the FAIR^2 Croissant package for the self-associating amphiphiles dataset.

Key observations:

- The dataset metadata is rich and includes descriptive information, provenance, funding, distributions, keywords, biases, limitations, and intended use cases.
- The provided metadata snapshot currently exposes `recordSet` as an empty list, so no record-level tables were directly loaded in this run.
- The notebook was written defensively so that if future versions of the Croissant package expose record sets and fields, the same workflow will automatically inspect them by `@id`, load them, and perform EDA.
- Even without record sets, package-level analysis still provides useful insight into dataset scope, documentation quality, and reuse boundaries.

If a future Croissant release includes populated record sets, the same notebook can be reused to:

- inspect each record set by its `@id`,
- enumerate field `@id` values,
- load records into pandas DataFrames,
- filter and normalize numeric variables,
- group observations by categorical fields, and
- generate record-level visualizations.
